In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "feature_engineered_data.csv",
    parse_dates=["time"]
)
df = df.sort_values("time").reset_index(drop=True)
print("Shape:", df.shape)

Shape: (377719, 44)


In [7]:
sensor_cols = [
    "Cyclone_Inlet_Gas_Temp",
    "Cyclone_Gas_Outlet_Temp",
    "Cyclone_Outlet_Gas_draft",
    "Cyclone_cone_draft",
    "Cyclone_Inlet_Draft",
    "Cyclone_Material_Temp"
]

relationship_features = [
    "Temp_Inlet_Outlet_Diff",
    "Temp_Material_Inlet_Diff",
    "Temp_Material_Outlet_Diff",
    "Draft_Inlet_Outlet_Diff",
    "Draft_Cone_Inlet_Diff",
    "Draft_Cone_Outlet_Diff"
]

change_features = [
    col for col in df.columns
    if "_change_5min" in col
    or "_change_30min" in col
    or "_change_60min" in col
]

rolling_std_features = [
    col for col in df.columns
    if "_rolling_std_60min" in col
]

model_features = (
    sensor_cols
    + relationship_features
    + change_features
    + rolling_std_features
)

print("Number of model features:", len(model_features))

Number of model features: 36


In [8]:
sensor_cols = [
    "Cyclone_Inlet_Gas_Temp",
    "Cyclone_Gas_Outlet_Temp",
    "Cyclone_Outlet_Gas_draft",
    "Cyclone_cone_draft",
    "Cyclone_Inlet_Draft",
    "Cyclone_Material_Temp"
]

relationship_features = [
    "Temp_Inlet_Outlet_Diff",
    "Temp_Material_Inlet_Diff",
    "Temp_Material_Outlet_Diff",
    "Draft_Inlet_Outlet_Diff",
    "Draft_Cone_Inlet_Diff",
    "Draft_Cone_Outlet_Diff"
]

change_features = [
    col for col in df.columns
    if "_change_5min" in col
    or "_change_30min" in col
    or "_change_60min" in col
]

rolling_std_features = [
    col for col in df.columns
    if "_rolling_std_60min" in col
]

model_features = (
    sensor_cols
    + relationship_features
    + change_features
    + rolling_std_features
)

print("Number of model features:", len(model_features))

Number of model features: 36


In [9]:
model_data = df.dropna(subset=model_features).copy()

X = model_data[model_features]

print("Original rows:", len(df))
print("Modeling rows:", len(model_data))
print("Features:", X.shape[1])
print("Missing values:", X.isna().sum().sum())
print("Infinite values:", np.isinf(X).sum().sum())

Original rows: 377719
Modeling rows: 374761
Features: 36
Missing values: 0
Infinite values: 0


In [10]:
from sklearn.ensemble import IsolationForest

In [11]:
iso_model = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

iso_model.fit(X)

print("Isolation Forest fitted successfully.")

Isolation Forest fitted successfully.


In [12]:
# Higher values from this score mean observations are more abnormal.
anomaly_score = -iso_model.score_samples(X)
print("Minimum score:", anomaly_score.min())
print("Maximum score:", anomaly_score.max())
print("Median score:", np.median(anomaly_score))

Minimum score: 0.3378433081847462
Maximum score: 0.7406175560827891
Median score: 0.3792851444011179


In [13]:
print(
    pd.Series(anomaly_score).describe(
        percentiles=[0.90, 0.95, 0.97, 0.98, 0.99, 0.995, 0.999]
    )
)

count    374761.000000
mean          0.394409
std           0.047201
min           0.337843
90%           0.448986
95%           0.485814
97%           0.519798
98%           0.547889
99%           0.597745
99.5%         0.637236
99.9%         0.685557
max           0.740618
dtype: float64


In [14]:
model_data["anomaly_score"] = anomaly_score
print(
    model_data[
        ["time", "anomaly_score"]
    ].sort_values("anomaly_score", ascending=False).head(20)
)

                      time  anomaly_score
199941 2018-11-27 05:40:00       0.740618
72351  2017-09-09 05:15:00       0.737177
199942 2018-11-27 05:45:00       0.728122
116791 2018-02-11 12:30:00       0.725777
199946 2018-11-27 06:05:00       0.725355
116793 2018-02-11 12:40:00       0.725066
116792 2018-02-11 12:35:00       0.724147
141844 2018-05-09 12:15:00       0.723763
148264 2018-05-31 19:15:00       0.723424
143966 2018-05-16 21:05:00       0.722930
199945 2018-11-27 06:00:00       0.722905
99625  2017-12-12 22:05:00       0.722345
199944 2018-11-27 05:55:00       0.722074
141843 2018-05-09 12:10:00       0.722027
143963 2018-05-16 20:50:00       0.720957
156184 2018-06-28 07:15:00       0.720677
143967 2018-05-16 21:10:00       0.720660
143964 2018-05-16 20:55:00       0.720127
143850 2018-05-16 11:25:00       0.719836
44258  2017-06-03 16:10:00       0.719391


In [15]:
thresholds = {
    "95th percentile": np.percentile(anomaly_score, 95),
    "97th percentile": np.percentile(anomaly_score, 97),
    "98th percentile": np.percentile(anomaly_score, 98),
    "99th percentile": np.percentile(anomaly_score, 99),
    "99.5th percentile": np.percentile(anomaly_score, 99.5),
    "99.9th percentile": np.percentile(anomaly_score, 99.9),
}

for name, threshold in thresholds.items():
    count = (anomaly_score >= threshold).sum()
    percentage = count / len(anomaly_score) * 100

    print(
        f"{name}: "
        f"threshold={threshold:.4f}, "
        f"observations={count}, "
        f"percentage={percentage:.2f}%"
    )

95th percentile: threshold=0.4858, observations=18739, percentage=5.00%
97th percentile: threshold=0.5198, observations=11243, percentage=3.00%
98th percentile: threshold=0.5479, observations=7496, percentage=2.00%
99th percentile: threshold=0.5977, observations=3748, percentage=1.00%
99.5th percentile: threshold=0.6372, observations=1874, percentage=0.50%
99.9th percentile: threshold=0.6856, observations=375, percentage=0.10%


testing persistence at several thresholds

In [16]:
# Add candidate anomaly flags
for name, threshold in thresholds.items():
    short_name = (
        name.replace("th percentile", "")
            .replace(".", "_")
            .replace("%", "")
    )
    model_data[f"anomaly_{short_name}"] = (
        model_data["anomaly_score"] >= threshold
    )

In [17]:

# Use the 99th percentile as the first candidate threshold
threshold_99 = thresholds["99th percentile"]
model_data["anomaly_99"] = (
    model_data["anomaly_score"] >= threshold_99
)

# Identify consecutive groups of anomalous/non-anomalous observations
model_data["anomaly_group"] = (
    model_data["anomaly_99"] != model_data["anomaly_99"].shift()
).cumsum()

groups_99 = (
    model_data[model_data["anomaly_99"]]
    .groupby("anomaly_group")
    .agg(
        start_time=("time", "min"),
        end_time=("time", "max"),
        observations=("time", "size"),
        max_score=("anomaly_score", "max"),
        mean_score=("anomaly_score", "mean")
    )
    .sort_values("observations", ascending=False)
)

print(groups_99.head(20))

                       start_time            end_time  observations  \
anomaly_group                                                         
720           2018-02-28 19:00:00 2018-02-28 21:00:00            25   
1826          2020-07-19 04:45:00 2020-07-19 06:30:00            22   
1342          2018-12-10 18:40:00 2018-12-10 20:20:00            21   
874           2018-05-16 20:45:00 2018-05-16 22:25:00            21   
250           2017-06-01 08:15:00 2017-06-01 09:45:00            19   
252           2017-06-03 16:05:00 2017-06-03 17:30:00            18   
1082          2018-07-10 13:35:00 2018-07-10 15:00:00            18   
1008          2018-06-19 23:25:00 2018-06-20 00:50:00            18   
870           2018-05-16 10:55:00 2018-05-16 12:10:00            16   
522           2017-12-12 23:30:00 2017-12-13 00:45:00            16   
1802          2020-05-21 11:20:00 2020-05-21 12:35:00            16   
702           2018-02-24 09:10:00 2018-02-24 10:25:00            16   
536   

In [18]:
threshold_99 = thresholds["99th percentile"]
model_data["anomaly_99"] = (
    model_data["anomaly_score"] >= threshold_99
)
time_gap = model_data["time"].diff()
new_group = (
    model_data["anomaly_99"].ne(model_data["anomaly_99"].shift())
    | time_gap.ne(pd.Timedelta(minutes=5))
)

model_data["anomaly_group"] = new_group.cumsum()

In [19]:
groups_99 = (
    model_data[model_data["anomaly_99"]]
    .groupby("anomaly_group")
    .agg(
        start_time=("time", "min"),
        end_time=("time", "max"),
        observations=("time", "size"),
        max_score=("anomaly_score", "max"),
        mean_score=("anomaly_score", "mean")
    )
    .sort_values(
        ["observations", "mean_score"],
        ascending=False
    )
)

groups_99.head(20)

,start_time,end_time,observations,max_score,mean_score
anomaly_group,,,,,
1027,2018-02-28 19:00:00,2018-02-28 21:00:00,25,0.696930,0.648336
2736,2020-07-19 04:45:00,2020-07-19 06:30:00,22,0.706428,0.664764
1254,2018-05-16 20:45:00,2018-05-16 22:25:00,21,0.722930,0.670374
1886,2018-12-10 18:40:00,2018-12-10 20:20:00,21,0.714939,0.659558
334,2017-06-01 08:15:00,2017-06-01 09:45:00,19,0.692856,0.645131
339,2017-06-03 16:05:00,2017-06-03 17:30:00,18,0.719391,0.661224
1421,2018-06-19 23:25:00,2018-06-20 00:50:00,18,0.676919,0.642903
1495,2018-07-10 13:35:00,2018-07-10 15:00:00,18,0.690323,0.631263
1250,2018-05-16 10:55:00,2018-05-16 12:10:00,16,0.719836,0.671819


In [20]:
id="w7q1pz"
start = pd.Timestamp("2017-06-09 02:15:00")
end = pd.Timestamp("2017-06-09 03:25:00")

period = model_data[
    (model_data["time"] >= start) &
    (model_data["time"] <= end)
].copy()
period[sensor_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Cyclone_Inlet_Gas_Temp,15.0,863.758667,119.383497,613.08,816.670,920.65,935.955,1004.08
Cyclone_Gas_Outlet_Temp,15.0,805.889333,109.927666,604.11,741.675,867.18,882.205,908.67
Cyclone_Outlet_Gas_draft,15.0,-101.567333,23.704631,-134.90,-123.050,-94.19,-88.895,-47.35
Cyclone_cone_draft,15.0,-130.734667,25.996875,-170.75,-149.325,-130.30,-117.400,-79.70
Cyclone_Inlet_Draft,15.0,-78.370667,17.272607,-108.35,-92.175,-74.02,-70.235,-42.61
Cyclone_Material_Temp,15.0,395.772667,394.435183,-96.30,45.220,161.57,800.000,800.00


In [21]:
id="r2v8kx"
period[sensor_cols].agg(["min", "max", "mean", "median"]).T

,min,max,mean,median
Cyclone_Inlet_Gas_Temp,613.08,1004.08,863.758667,920.65
Cyclone_Gas_Outlet_Temp,604.11,908.67,805.889333,867.18
Cyclone_Outlet_Gas_draft,-134.90,-47.35,-101.567333,-94.19
Cyclone_cone_draft,-170.75,-79.70,-130.734667,-130.30
Cyclone_Inlet_Draft,-108.35,-42.61,-78.370667,-74.02
Cyclone_Material_Temp,-96.30,800.00,395.772667,161.57


In [22]:
id="h4m6cs"
period[
    ["time"] +
    sensor_cols +
    ["anomaly_score"]
]

,time,Cyclone_Inlet_Gas_Temp,Cyclone_Gas_Outlet_Temp,Cyclone_Outlet_Gas_draft,Cyclone_cone_draft,Cyclone_Inlet_Draft,Cyclone_Material_Temp,anomaly_score
45819,2017-06-09 02:15:00,613.08,604.11,-47.35,-79.70,-42.61,45.22,0.654321
45820,2017-06-09 02:20:00,651.40,613.60,-87.20,-125.54,-73.16,45.22,0.683681
45821,2017-06-09 02:25:00,707.16,637.45,-88.37,-116.07,-74.02,-96.30,0.685681
45822,2017-06-09 02:30:00,784.18,725.17,-86.44,-98.18,-62.78,45.22,0.716771
45823,2017-06-09 02:35:00,849.16,758.18,-93.63,-118.73,-70.55,45.22,0.714339
45824,2017-06-09 02:40:00,925.46,817.38,-93.27,-99.37,-74.21,45.22,0.717103
45825,2017-06-09 02:45:00,916.69,843.59,-89.42,-135.01,-68.11,45.22,0.713774
45826,2017-06-09 02:50:00,952.13,877.05,-96.22,-129.05,-69.92,161.57,0.705586
45827,2017-06-09 02:55:00,921.85,879.44,-94.19,-130.30,-70.65,800.00,0.678795
45828,2017-06-09 03:00:00,958.77,884.18,-102.32,-136.03,-76.57,800.00,0.665494


In [23]:
start = pd.Timestamp("2018-02-28 19:00:00")
end = pd.Timestamp("2018-02-28 21:00:00")

period = model_data[
    (model_data["time"] >= start) &
    (model_data["time"] <= end)
].copy()

period[sensor_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Cyclone_Inlet_Gas_Temp,25.0,904.3404,83.638316,803.10,826.33,889.09,977.14,1076.39
Cyclone_Gas_Outlet_Temp,25.0,871.7780,68.570184,777.84,810.82,847.46,930.94,1014.20
Cyclone_Outlet_Gas_draft,25.0,-135.4968,86.313567,-277.43,-195.56,-128.04,-78.26,19.38
Cyclone_cone_draft,25.0,-150.4492,86.164608,-293.17,-207.14,-142.27,-98.43,8.70
Cyclone_Inlet_Draft,25.0,-113.1052,73.788381,-248.53,-170.52,-98.23,-67.98,18.99
Cyclone_Material_Temp,25.0,976.5228,45.352355,850.77,955.61,980.37,1017.24,1026.87


In [24]:
period[sensor_cols].agg(
    ["min", "max", "mean", "median"]
).T

,min,max,mean,median
Cyclone_Inlet_Gas_Temp,803.10,1076.39,904.3404,889.09
Cyclone_Gas_Outlet_Temp,777.84,1014.20,871.7780,847.46
Cyclone_Outlet_Gas_draft,-277.43,19.38,-135.4968,-128.04
Cyclone_cone_draft,-293.17,8.70,-150.4492,-142.27
Cyclone_Inlet_Draft,-248.53,18.99,-113.1052,-98.23
Cyclone_Material_Temp,850.77,1026.87,976.5228,980.37


In [25]:
period[
    ["time"] + sensor_cols + ["anomaly_score"]
]

,time,Cyclone_Inlet_Gas_Temp,Cyclone_Gas_Outlet_Temp,Cyclone_Outlet_Gas_draft,Cyclone_cone_draft,Cyclone_Inlet_Draft,Cyclone_Material_Temp,anomaly_score
121765,2018-02-28 19:00:00,803.10,777.84,-116.33,-129.50,-93.71,941.96,0.660112
121766,2018-02-28 19:05:00,816.76,805.75,2.00,-10.69,2.11,906.45,0.696930
121767,2018-02-28 19:10:00,818.20,810.13,8.09,-2.65,4.63,931.33,0.668874
121768,2018-02-28 19:15:00,815.78,810.82,19.38,8.70,18.99,964.16,0.659023
121769,2018-02-28 19:20:00,812.79,806.42,-4.82,-19.41,-6.87,980.37,0.680916
121770,2018-02-28 19:25:00,808.65,800.73,-78.26,-104.42,-67.98,978.94,0.688934
121771,2018-02-28 19:30:00,889.09,873.15,-56.97,-83.79,-45.71,987.26,0.648753
121772,2018-02-28 19:35:00,988.81,847.46,-72.43,-86.61,-55.75,1019.59,0.659022
121773,2018-02-28 19:40:00,910.48,844.49,-84.28,-98.43,-72.40,1023.88,0.642363
121774,2018-02-28 19:45:00,977.14,909.32,-104.32,-122.80,-89.84,1023.64,0.644342


In [26]:
min_observations = 6   # 6 × 5 minutes = 30 minutes

final_periods = groups_99[
    groups_99["observations"] >= min_observations
].copy()

print("Number of abnormal periods:", len(final_periods))
print()
print(final_periods.head(20))

Number of abnormal periods: 252

                       start_time            end_time  observations  \
anomaly_group                                                         
1027          2018-02-28 19:00:00 2018-02-28 21:00:00            25   
2736          2020-07-19 04:45:00 2020-07-19 06:30:00            22   
1254          2018-05-16 20:45:00 2018-05-16 22:25:00            21   
1886          2018-12-10 18:40:00 2018-12-10 20:20:00            21   
334           2017-06-01 08:15:00 2017-06-01 09:45:00            19   
339           2017-06-03 16:05:00 2017-06-03 17:30:00            18   
1421          2018-06-19 23:25:00 2018-06-20 00:50:00            18   
1495          2018-07-10 13:35:00 2018-07-10 15:00:00            18   
1250          2018-05-16 10:55:00 2018-05-16 12:10:00            16   
1008          2018-02-24 09:10:00 2018-02-24 10:25:00            16   
2652          2020-05-21 11:20:00 2020-05-21 12:35:00            16   
788           2017-12-12 23:30:00 2017-12-13

In [27]:
final_periods = final_periods.reset_index(drop=True)
final_periods["duration"] = (
    final_periods["end_time"] - final_periods["start_time"]
)
final_periods = final_periods[
    [
        "start_time",
        "end_time",
        "duration",
        "observations",
        "max_score",
        "mean_score"
    ]
]

final_periods.head(20)

,start_time,end_time,duration,observations,max_score,mean_score
0,2018-02-28 19:00:00,2018-02-28 21:00:00,0 days 02:00:00,25,0.696930,0.648336
1,2020-07-19 04:45:00,2020-07-19 06:30:00,0 days 01:45:00,22,0.706428,0.664764
2,2018-05-16 20:45:00,2018-05-16 22:25:00,0 days 01:40:00,21,0.722930,0.670374
3,2018-12-10 18:40:00,2018-12-10 20:20:00,0 days 01:40:00,21,0.714939,0.659558
4,2017-06-01 08:15:00,2017-06-01 09:45:00,0 days 01:30:00,19,0.692856,0.645131
5,2017-06-03 16:05:00,2017-06-03 17:30:00,0 days 01:25:00,18,0.719391,0.661224
6,2018-06-19 23:25:00,2018-06-20 00:50:00,0 days 01:25:00,18,0.676919,0.642903
7,2018-07-10 13:35:00,2018-07-10 15:00:00,0 days 01:25:00,18,0.690323,0.631263
8,2018-05-16 10:55:00,2018-05-16 12:10:00,0 days 01:15:00,16,0.719836,0.671819
9,2018-02-24 09:10:00,2018-02-24 10:25:00,0 days 01:15:00,16,0.705654,0.656666


In [28]:
final_periods.to_csv(
    "detected_abnormal_periods.csv",
    index=False
)

In [29]:
period = df[
    (df["time"] >= "2018-02-28 19:00:00") &
    (df["time"] <= "2018-02-28 21:00:00")
]

period[
    [
        "time",
        "Cyclone_Inlet_Gas_Temp",
        "Cyclone_Gas_Outlet_Temp",
        "Cyclone_Outlet_Gas_draft",
        "Cyclone_cone_draft",
        "Cyclone_Inlet_Draft",
        "Cyclone_Material_Temp"
    ]
]

,time,Cyclone_Inlet_Gas_Temp,Cyclone_Gas_Outlet_Temp,Cyclone_Outlet_Gas_draft,Cyclone_cone_draft,Cyclone_Inlet_Draft,Cyclone_Material_Temp
121765,2018-02-28 19:00:00,803.10,777.84,-116.33,-129.50,-93.71,941.96
121766,2018-02-28 19:05:00,816.76,805.75,2.00,-10.69,2.11,906.45
121767,2018-02-28 19:10:00,818.20,810.13,8.09,-2.65,4.63,931.33
121768,2018-02-28 19:15:00,815.78,810.82,19.38,8.70,18.99,964.16
121769,2018-02-28 19:20:00,812.79,806.42,-4.82,-19.41,-6.87,980.37
121770,2018-02-28 19:25:00,808.65,800.73,-78.26,-104.42,-67.98,978.94
121771,2018-02-28 19:30:00,889.09,873.15,-56.97,-83.79,-45.71,987.26
121772,2018-02-28 19:35:00,988.81,847.46,-72.43,-86.61,-55.75,1019.59
121773,2018-02-28 19:40:00,910.48,844.49,-84.28,-98.43,-72.40,1023.88
121774,2018-02-28 19:45:00,977.14,909.32,-104.32,-122.80,-89.84,1023.64


In [30]:
final_periods[
    [
        "start_time",
        "end_time",
        "duration",
        "observations",
        "max_score",
        "mean_score"
    ]
].to_string(index=False)

'         start_time            end_time        duration  observations  max_score  mean_score\n2018-02-28 19:00:00 2018-02-28 21:00:00 0 days 02:00:00            25   0.696930    0.648336\n2020-07-19 04:45:00 2020-07-19 06:30:00 0 days 01:45:00            22   0.706428    0.664764\n2018-05-16 20:45:00 2018-05-16 22:25:00 0 days 01:40:00            21   0.722930    0.670374\n2018-12-10 18:40:00 2018-12-10 20:20:00 0 days 01:40:00            21   0.714939    0.659558\n2017-06-01 08:15:00 2017-06-01 09:45:00 0 days 01:30:00            19   0.692856    0.645131\n2017-06-03 16:05:00 2017-06-03 17:30:00 0 days 01:25:00            18   0.719391    0.661224\n2018-06-19 23:25:00 2018-06-20 00:50:00 0 days 01:25:00            18   0.676919    0.642903\n2018-07-10 13:35:00 2018-07-10 15:00:00 0 days 01:25:00            18   0.690323    0.631263\n2018-05-16 10:55:00 2018-05-16 12:10:00 0 days 01:15:00            16   0.719836    0.671819\n2018-02-24 09:10:00 2018-02-24 10:25:00 0 days 01:15:00    

In [31]:
final_periods["year"] = final_periods["start_time"].dt.year

final_periods["year"].value_counts().sort_index()

year
2017     80
2018    102
2019     49
2020     21
Name: count, dtype: int64

In [32]:
final_periods.groupby("year")["duration"].sum()

year
2017   2 days 14:30:00
2018   3 days 07:10:00
2019   1 days 12:35:00
2020   0 days 17:40:00
Name: duration, dtype: timedelta64[us]

In [33]:
final_periods[
    [
        "start_time",
        "end_time",
        "duration",
        "observations",
        "max_score",
        "mean_score"
    ]
].to_string(index=False)

'         start_time            end_time        duration  observations  max_score  mean_score\n2018-02-28 19:00:00 2018-02-28 21:00:00 0 days 02:00:00            25   0.696930    0.648336\n2020-07-19 04:45:00 2020-07-19 06:30:00 0 days 01:45:00            22   0.706428    0.664764\n2018-05-16 20:45:00 2018-05-16 22:25:00 0 days 01:40:00            21   0.722930    0.670374\n2018-12-10 18:40:00 2018-12-10 20:20:00 0 days 01:40:00            21   0.714939    0.659558\n2017-06-01 08:15:00 2017-06-01 09:45:00 0 days 01:30:00            19   0.692856    0.645131\n2017-06-03 16:05:00 2017-06-03 17:30:00 0 days 01:25:00            18   0.719391    0.661224\n2018-06-19 23:25:00 2018-06-20 00:50:00 0 days 01:25:00            18   0.676919    0.642903\n2018-07-10 13:35:00 2018-07-10 15:00:00 0 days 01:25:00            18   0.690323    0.631263\n2018-05-16 10:55:00 2018-05-16 12:10:00 0 days 01:15:00            16   0.719836    0.671819\n2018-02-24 09:10:00 2018-02-24 10:25:00 0 days 01:15:00    

In [34]:
final_periods["year"] = final_periods["start_time"].dt.year

final_periods["year"].value_counts().sort_index()

year
2017     80
2018    102
2019     49
2020     21
Name: count, dtype: int64

10 strongest sustained abnormal periods

In [35]:
final_periods.sort_values(
    "mean_score",
    ascending=False
).head(10)

,start_time,end_time,duration,observations,max_score,mean_score,year
79,2018-11-27 05:40:00,2018-11-27 06:30:00,0 days 00:50:00,11,0.740618,0.703044,2018
34,2018-02-11 12:15:00,2018-02-11 13:10:00,0 days 00:55:00,12,0.725777,0.694199,2018
152,2018-05-09 12:05:00,2018-05-09 12:45:00,0 days 00:40:00,9,0.723763,0.691307,2018
26,2018-09-15 08:35:00,2018-09-15 09:35:00,0 days 01:00:00,13,0.713336,0.687895,2018
35,2018-05-31 19:10:00,2018-05-31 20:05:00,0 days 00:55:00,12,0.723424,0.685048,2018
36,2018-08-22 18:10:00,2018-08-22 19:05:00,0 days 00:55:00,12,0.714466,0.683196,2018
80,2018-07-15 01:45:00,2018-07-15 02:35:00,0 days 00:50:00,11,0.714736,0.681778,2018
37,2017-12-12 22:00:00,2017-12-12 22:55:00,0 days 00:55:00,12,0.722345,0.681088,2017
81,2019-02-25 14:50:00,2019-02-25 15:40:00,0 days 00:50:00,11,0.714483,0.680432,2019
38,2017-06-08 16:30:00,2017-06-08 17:25:00,0 days 00:55:00,12,0.701223,0.678640,2017


In [36]:
period = df[
    (df["time"] >= "2018-11-27 05:40:00") &
    (df["time"] <= "2018-11-27 06:30:00")
]

sensor_cols = [
    "Cyclone_Inlet_Gas_Temp",
    "Cyclone_Gas_Outlet_Temp",
    "Cyclone_Outlet_Gas_draft",
    "Cyclone_cone_draft",
    "Cyclone_Inlet_Draft",
    "Cyclone_Material_Temp"
]

period[sensor_cols].agg(
    ["mean", "std", "min", "max"]
).T

,mean,std,min,max
Cyclone_Inlet_Gas_Temp,750.469091,13.691761,731.14,777.18
Cyclone_Gas_Outlet_Temp,716.136364,10.987535,702.10,740.82
Cyclone_Outlet_Gas_draft,19.517273,7.954751,-4.27,23.52
Cyclone_cone_draft,-4.230000,7.426220,-26.62,-1.82
Cyclone_Inlet_Draft,16.991818,5.833522,-0.38,20.05
Cyclone_Material_Temp,547.530909,43.641036,490.26,640.44


In [37]:
# Create a readable summary of the detected abnormal periods

sensor_cols = [
    "Cyclone_Inlet_Gas_Temp",
    "Cyclone_Gas_Outlet_Temp",
    "Cyclone_Outlet_Gas_draft",
    "Cyclone_cone_draft",
    "Cyclone_Inlet_Draft",
    "Cyclone_Material_Temp"
]

# Overall normal reference statistics
normal_stats = df[sensor_cols].agg(["mean", "std"])

results = []

for _, row in final_periods.iterrows():

    period = df[
        (df["time"] >= row["start_time"]) &
        (df["time"] <= row["end_time"])
    ]

    abnormal_sensors = []

    for sensor in sensor_cols:
        period_mean = period[sensor].mean()
        normal_mean = normal_stats.loc["mean", sensor]
        normal_std = normal_stats.loc["std", sensor]

        if normal_std > 0:
            deviation = abs(period_mean - normal_mean) / normal_std

            if deviation >= 2:
                abnormal_sensors.append(sensor)

    results.append({
        "start_time": row["start_time"],
        "end_time": row["end_time"],
        "duration": row["duration"],
        "mean_anomaly_score": row["mean_score"],
        "max_anomaly_score": row["max_score"],
        "abnormal_sensors": ", ".join(abnormal_sensors)
    })

abnormal_period_report = pd.DataFrame(results)

abnormal_period_report.head(20)

,start_time,end_time,duration,mean_anomaly_score,max_anomaly_score,abnormal_sensors
0,2018-02-28 19:00:00,2018-02-28 21:00:00,0 days 02:00:00,0.648336,0.696930,
1,2020-07-19 04:45:00,2020-07-19 06:30:00,0 days 01:45:00,0.664764,0.706428,
2,2018-05-16 20:45:00,2018-05-16 22:25:00,0 days 01:40:00,0.670374,0.722930,
3,2018-12-10 18:40:00,2018-12-10 20:20:00,0 days 01:40:00,0.659558,0.714939,
4,2017-06-01 08:15:00,2017-06-01 09:45:00,0 days 01:30:00,0.645131,0.692856,
5,2017-06-03 16:05:00,2017-06-03 17:30:00,0 days 01:25:00,0.661224,0.719391,
6,2018-06-19 23:25:00,2018-06-20 00:50:00,0 days 01:25:00,0.642903,0.676919,
7,2018-07-10 13:35:00,2018-07-10 15:00:00,0 days 01:25:00,0.631263,0.690323,
8,2018-05-16 10:55:00,2018-05-16 12:10:00,0 days 01:15:00,0.671819,0.719836,
9,2018-02-24 09:10:00,2018-02-24 10:25:00,0 days 01:15:00,0.656666,0.705654,


In [38]:
# Create a simple automatic description for each abnormal period

def describe_period(row, df):
    period = df[
        (df["time"] >= row["start_time"]) &
        (df["time"] <= row["end_time"])
    ]

    descriptions = []
    # Temperature behavior
    inlet_temp_range = period["Cyclone_Inlet_Gas_Temp"].max() - period["Cyclone_Inlet_Gas_Temp"].min()
    outlet_temp_range = period["Cyclone_Gas_Outlet_Temp"].max() - period["Cyclone_Gas_Outlet_Temp"].min()

    if inlet_temp_range > 150 or outlet_temp_range > 150:
        descriptions.append("large temperature variation")
    outlet_draft_range = (
        period["Cyclone_Outlet_Gas_draft"].max()
        - period["Cyclone_Outlet_Gas_draft"].min()
    )

    cone_draft_range = (
        period["Cyclone_cone_draft"].max()
        - period["Cyclone_cone_draft"].min()
    )

    if outlet_draft_range > 150 or cone_draft_range > 150:
        descriptions.append("large draft variation")
    material_range = (
        period["Cyclone_Material_Temp"].max()
        - period["Cyclone_Material_Temp"].min()
    )

    if material_range > 150:
        descriptions.append("large material-temperature variation")
    if not descriptions:
        descriptions.append("unusual multivariate process behavior")
    return ", ".join(descriptions)

final_periods["behavior"] = final_periods.apply(
    lambda row: describe_period(row, model_data),
    axis=1
)

final_periods.head(20)

,start_time,end_time,duration,observations,max_score,mean_score,year,behavior
0,2018-02-28 19:00:00,2018-02-28 21:00:00,0 days 02:00:00,25,0.696930,0.648336,2018,"large temperature variation, large draft varia..."
1,2020-07-19 04:45:00,2020-07-19 06:30:00,0 days 01:45:00,22,0.706428,0.664764,2020,"large temperature variation, large draft varia..."
2,2018-05-16 20:45:00,2018-05-16 22:25:00,0 days 01:40:00,21,0.722930,0.670374,2018,"large temperature variation, large draft varia..."
3,2018-12-10 18:40:00,2018-12-10 20:20:00,0 days 01:40:00,21,0.714939,0.659558,2018,"large temperature variation, large draft varia..."
4,2017-06-01 08:15:00,2017-06-01 09:45:00,0 days 01:30:00,19,0.692856,0.645131,2017,"large temperature variation, large draft varia..."
5,2017-06-03 16:05:00,2017-06-03 17:30:00,0 days 01:25:00,18,0.719391,0.661224,2017,"large temperature variation, large draft varia..."
6,2018-06-19 23:25:00,2018-06-20 00:50:00,0 days 01:25:00,18,0.676919,0.642903,2018,"large temperature variation, large draft varia..."
7,2018-07-10 13:35:00,2018-07-10 15:00:00,0 days 01:25:00,18,0.690323,0.631263,2018,"large temperature variation, large material-te..."
8,2018-05-16 10:55:00,2018-05-16 12:10:00,0 days 01:15:00,16,0.719836,0.671819,2018,"large temperature variation, large material-te..."
9,2018-02-24 09:10:00,2018-02-24 10:25:00,0 days 01:15:00,16,0.705654,0.656666,2018,"large temperature variation, large draft varia..."


final output


In [39]:
final_periods.to_csv(
    "detected_abnormal_periods.csv",
    index=False
)

print("Detected abnormal periods:", len(final_periods))
print("Saved to: detected_abnormal_periods.csv")

Detected abnormal periods: 252
Saved to: detected_abnormal_periods.csv


In [40]:
final_periods[
    [
        "start_time",
        "end_time",
        "duration",
        "mean_score",
        "max_score",
        "behavior"
    ]
].head(20)

,start_time,end_time,duration,mean_score,max_score,behavior
0,2018-02-28 19:00:00,2018-02-28 21:00:00,0 days 02:00:00,0.648336,0.696930,"large temperature variation, large draft varia..."
1,2020-07-19 04:45:00,2020-07-19 06:30:00,0 days 01:45:00,0.664764,0.706428,"large temperature variation, large draft varia..."
2,2018-05-16 20:45:00,2018-05-16 22:25:00,0 days 01:40:00,0.670374,0.722930,"large temperature variation, large draft varia..."
3,2018-12-10 18:40:00,2018-12-10 20:20:00,0 days 01:40:00,0.659558,0.714939,"large temperature variation, large draft varia..."
4,2017-06-01 08:15:00,2017-06-01 09:45:00,0 days 01:30:00,0.645131,0.692856,"large temperature variation, large draft varia..."
5,2017-06-03 16:05:00,2017-06-03 17:30:00,0 days 01:25:00,0.661224,0.719391,"large temperature variation, large draft varia..."
6,2018-06-19 23:25:00,2018-06-20 00:50:00,0 days 01:25:00,0.642903,0.676919,"large temperature variation, large draft varia..."
7,2018-07-10 13:35:00,2018-07-10 15:00:00,0 days 01:25:00,0.631263,0.690323,"large temperature variation, large material-te..."
8,2018-05-16 10:55:00,2018-05-16 12:10:00,0 days 01:15:00,0.671819,0.719836,"large temperature variation, large material-te..."
9,2018-02-24 09:10:00,2018-02-24 10:25:00,0 days 01:15:00,0.656666,0.705654,"large temperature variation, large draft varia..."


overall result


In [ ]:
print("Total abnormal periods:", len(final_periods))
print("\nPeriods by year:")
print(
    final_periods["start_time"]
    .dt.year
    .value_counts()
    .sort_index()
)

print("\nTotal abnormal duration by year:")
year_duration = (
    final_periods
    .assign(year=final_periods["start_time"].dt.year)
    .groupby("year")["duration"]
    .sum()
)

print(year_duration)

print("\nOverall abnormal duration:")
print(final_periods["duration"].sum())

Total abnormal periods: 252

Periods by year:
start_time
2017     80
2018    102
2019     49
2020     21
Name: count, dtype: int64

Total abnormal duration by year:
year
2017   2 days 14:30:00
2018   3 days 07:10:00
2019   1 days 12:35:00
2020   0 days 17:40:00
Name: duration, dtype: timedelta64[us]

Overall abnormal duration:
8 days 03:55:00


In [44]:
print(final_periods.columns.tolist())

['start_time', 'end_time', 'duration', 'observations', 'max_score', 'mean_score', 'year', 'behavior']


The model identifying the strongest periods

In [45]:
top_10 = (
    final_periods
    .sort_values("mean_score", ascending=False)
    .head(10)
)

top_10[
    [
        "start_time",
        "end_time",
        "duration",
        "mean_score",
        "max_score"
    ]
]

,start_time,end_time,duration,mean_score,max_score
79,2018-11-27 05:40:00,2018-11-27 06:30:00,0 days 00:50:00,0.703044,0.740618
34,2018-02-11 12:15:00,2018-02-11 13:10:00,0 days 00:55:00,0.694199,0.725777
152,2018-05-09 12:05:00,2018-05-09 12:45:00,0 days 00:40:00,0.691307,0.723763
26,2018-09-15 08:35:00,2018-09-15 09:35:00,0 days 01:00:00,0.687895,0.713336
35,2018-05-31 19:10:00,2018-05-31 20:05:00,0 days 00:55:00,0.685048,0.723424
36,2018-08-22 18:10:00,2018-08-22 19:05:00,0 days 00:55:00,0.683196,0.714466
80,2018-07-15 01:45:00,2018-07-15 02:35:00,0 days 00:50:00,0.681778,0.714736
37,2017-12-12 22:00:00,2017-12-12 22:55:00,0 days 00:55:00,0.681088,0.722345
81,2019-02-25 14:50:00,2019-02-25 15:40:00,0 days 00:50:00,0.680432,0.714483
38,2017-06-08 16:30:00,2017-06-08 17:25:00,0 days 00:55:00,0.678640,0.701223


In [46]:
top_10[
    [
        "start_time",
        "end_time",
        "duration",
        "mean_score",
        "max_score",
        "behavior"
    ]
]

,start_time,end_time,duration,mean_score,max_score,behavior
79,2018-11-27 05:40:00,2018-11-27 06:30:00,0 days 00:50:00,0.703044,0.740618,large material-temperature variation
34,2018-02-11 12:15:00,2018-02-11 13:10:00,0 days 00:55:00,0.694199,0.725777,"large draft variation, large material-temperat..."
152,2018-05-09 12:05:00,2018-05-09 12:45:00,0 days 00:40:00,0.691307,0.723763,large material-temperature variation
26,2018-09-15 08:35:00,2018-09-15 09:35:00,0 days 01:00:00,0.687895,0.713336,"large temperature variation, large draft varia..."
35,2018-05-31 19:10:00,2018-05-31 20:05:00,0 days 00:55:00,0.685048,0.723424,large material-temperature variation
36,2018-08-22 18:10:00,2018-08-22 19:05:00,0 days 00:55:00,0.683196,0.714466,"large temperature variation, large material-te..."
80,2018-07-15 01:45:00,2018-07-15 02:35:00,0 days 00:50:00,0.681778,0.714736,large material-temperature variation
37,2017-12-12 22:00:00,2017-12-12 22:55:00,0 days 00:55:00,0.681088,0.722345,"large draft variation, large material-temperat..."
81,2019-02-25 14:50:00,2019-02-25 15:40:00,0 days 00:50:00,0.680432,0.714483,large material-temperature variation
38,2017-06-08 16:30:00,2017-06-08 17:25:00,0 days 00:55:00,0.678640,0.701223,large material-temperature variation


In [47]:
# Select the strongest abnormal period
top_period = top_10.iloc[0]

start = top_period["start_time"]
end = top_period["end_time"]

# Get the abnormal period
abnormal = model_data[
    (model_data["time"] >= start) &
    (model_data["time"] <= end)
]

# Get 1 hour before and after for comparison
before = model_data[
    (model_data["time"] >= start - pd.Timedelta(hours=1)) &
    (model_data["time"] < start)
]

after = model_data[
    (model_data["time"] > end) &
    (model_data["time"] <= end + pd.Timedelta(hours=1))
]

print("ABNORMAL PERIOD")
print(start, "to", end)

print("\nSensor values during abnormal period:")
print(
    abnormal[sensor_cols]
    .agg(["mean", "min", "max"])
    .round(2)
)

ABNORMAL PERIOD
2018-11-27 05:40:00 to 2018-11-27 06:30:00

Sensor values during abnormal period:
      Cyclone_Inlet_Gas_Temp  Cyclone_Gas_Outlet_Temp  \
mean                  750.47                   716.14   
min                   731.14                   702.10   
max                   777.18                   740.82   

      Cyclone_Outlet_Gas_draft  Cyclone_cone_draft  Cyclone_Inlet_Draft  \
mean                     19.52               -4.23                16.99   
min                      -4.27              -26.62                -0.38   
max                      23.52               -1.82                20.05   

      Cyclone_Material_Temp  
mean                 547.53  
min                  490.26  
max                  640.44  


In [49]:
def describe_period_change(row, df):
    start = row["start_time"]
    end = row["end_time"]

    # Abnormal period
    abnormal = df[
        (df["time"] >= start) &
        (df["time"] <= end)
    ]

    # 1 hour before abnormal period
    before = df[
        (df["time"] >= start - pd.Timedelta(hours=1)) &
        (df["time"] < start)
    ]

    if abnormal.empty or before.empty:
        return "unusual multivariate process behavior"

    descriptions = []

    # Compare average values before vs during
    for sensor in sensor_cols:
        before_mean = before[sensor].mean()
        abnormal_mean = abnormal[sensor].mean()

        if pd.isna(before_mean) or pd.isna(abnormal_mean):
            continue

        change = abnormal_mean - before_mean

        # Use percentage change where appropriate
        if abs(before_mean) > 20:
            pct_change = abs(change / before_mean) * 100
        else:
            pct_change = 0

        # Temperature sensors
        if sensor in [
            "Cyclone_Inlet_Gas_Temp",
            "Cyclone_Gas_Outlet_Temp",
            "Cyclone_Material_Temp"
        ]:
            if pct_change >= 15:
                if change < 0:
                    descriptions.append(
                        f"{sensor.replace('Cyclone_', '').replace('_', ' ')} decrease"
                    )
                else:
                    descriptions.append(
                        f"{sensor.replace('Cyclone_', '').replace('_', ' ')} increase"
                    )

        # Draft sensors
        elif sensor in [
            "Cyclone_Outlet_Gas_draft",
            "Cyclone_cone_draft",
            "Cyclone_Inlet_Draft"
        ]:
            if abs(change) >= 50:
                descriptions.append(
                    f"{sensor.replace('Cyclone_', '').replace('_', ' ')} shift"
                )

    if not descriptions:
        return "unusual multivariate process behavior"

    # Keep maximum 3 descriptions so the table stays readable
    return " + ".join(descriptions[:3])


final_periods["behavior"] = final_periods.apply(
    lambda row: describe_period_change(row, model_data),
    axis=1
)

In [50]:
top_10 = (
    final_periods
    .sort_values("mean_score", ascending=False)
    .head(10)
)

top_10[
    [
        "start_time",
        "end_time",
        "duration",
        "mean_score",
        "max_score",
        "behavior"
    ]
]

,start_time,end_time,duration,mean_score,max_score,behavior
79,2018-11-27 05:40:00,2018-11-27 06:30:00,0 days 00:50:00,0.703044,0.740618,Inlet Gas Temp decrease + Gas Outlet Temp decr...
34,2018-02-11 12:15:00,2018-02-11 13:10:00,0 days 00:55:00,0.694199,0.725777,Inlet Gas Temp decrease + Gas Outlet Temp decr...
152,2018-05-09 12:05:00,2018-05-09 12:45:00,0 days 00:40:00,0.691307,0.723763,Inlet Gas Temp decrease + Gas Outlet Temp decr...
26,2018-09-15 08:35:00,2018-09-15 09:35:00,0 days 01:00:00,0.687895,0.713336,Inlet Gas Temp decrease + Gas Outlet Temp decr...
35,2018-05-31 19:10:00,2018-05-31 20:05:00,0 days 00:55:00,0.685048,0.723424,Inlet Gas Temp decrease + Gas Outlet Temp decr...
36,2018-08-22 18:10:00,2018-08-22 19:05:00,0 days 00:55:00,0.683196,0.714466,Outlet Gas draft shift + cone draft shift + In...
80,2018-07-15 01:45:00,2018-07-15 02:35:00,0 days 00:50:00,0.681778,0.714736,Outlet Gas draft shift + cone draft shift + In...
37,2017-12-12 22:00:00,2017-12-12 22:55:00,0 days 00:55:00,0.681088,0.722345,Outlet Gas draft shift + cone draft shift + In...
81,2019-02-25 14:50:00,2019-02-25 15:40:00,0 days 00:50:00,0.680432,0.714483,Inlet Gas Temp decrease + Outlet Gas draft shi...
38,2017-06-08 16:30:00,2017-06-08 17:25:00,0 days 00:55:00,0.678640,0.701223,Inlet Gas Temp decrease + Outlet Gas draft shi...


In [51]:
final_periods.to_csv(
    "detected_abnormal_periods.csv",
    index=False
)

print("Final abnormal periods:", len(final_periods))
print("File saved: detected_abnormal_periods.csv")

Final abnormal periods: 252
File saved: detected_abnormal_periods.csv


In [52]:
pd.set_option("display.max_colwidth", None)
top_10[
    [
        "start_time",
        "end_time",
        "duration",
        "mean_score",
        "max_score",
        "behavior"
    ]
]

,start_time,end_time,duration,mean_score,max_score,behavior
79,2018-11-27 05:40:00,2018-11-27 06:30:00,0 days 00:50:00,0.703044,0.740618,Inlet Gas Temp decrease + Gas Outlet Temp decrease + Outlet Gas draft shift
34,2018-02-11 12:15:00,2018-02-11 13:10:00,0 days 00:55:00,0.694199,0.725777,Inlet Gas Temp decrease + Gas Outlet Temp decrease + Outlet Gas draft shift
152,2018-05-09 12:05:00,2018-05-09 12:45:00,0 days 00:40:00,0.691307,0.723763,Inlet Gas Temp decrease + Gas Outlet Temp decrease + Outlet Gas draft shift
26,2018-09-15 08:35:00,2018-09-15 09:35:00,0 days 01:00:00,0.687895,0.713336,Inlet Gas Temp decrease + Gas Outlet Temp decrease + Outlet Gas draft shift
35,2018-05-31 19:10:00,2018-05-31 20:05:00,0 days 00:55:00,0.685048,0.723424,Inlet Gas Temp decrease + Gas Outlet Temp decrease + Outlet Gas draft shift
36,2018-08-22 18:10:00,2018-08-22 19:05:00,0 days 00:55:00,0.683196,0.714466,Outlet Gas draft shift + cone draft shift + Inlet Draft shift
80,2018-07-15 01:45:00,2018-07-15 02:35:00,0 days 00:50:00,0.681778,0.714736,Outlet Gas draft shift + cone draft shift + Inlet Draft shift
37,2017-12-12 22:00:00,2017-12-12 22:55:00,0 days 00:55:00,0.681088,0.722345,Outlet Gas draft shift + cone draft shift + Inlet Draft shift
81,2019-02-25 14:50:00,2019-02-25 15:40:00,0 days 00:50:00,0.680432,0.714483,Inlet Gas Temp decrease + Outlet Gas draft shift + cone draft shift
38,2017-06-08 16:30:00,2017-06-08 17:25:00,0 days 00:55:00,0.678640,0.701223,Inlet Gas Temp decrease + Outlet Gas draft shift + cone draft shift


In [53]:
final_periods.to_csv(
    "detected_abnormal_periods.csv",
    index=False
)

print("Total detected abnormal periods:", len(final_periods))

Total detected abnormal periods: 252


In [54]:
year_summary = (
    final_periods
    .assign(year=final_periods["start_time"].dt.year)
    .groupby("year")
    .agg(
        abnormal_periods=("start_time", "count"),
        total_duration=("duration", "sum"),
        highest_mean_score=("mean_score", "max"),
        highest_max_score=("max_score", "max")
    )
    .reset_index()
)

year_summary

,year,abnormal_periods,total_duration,highest_mean_score,highest_max_score
0,2017,80,2 days 14:30:00,0.681088,0.737177
1,2018,102,3 days 07:10:00,0.703044,0.740618
2,2019,49,1 days 12:35:00,0.680432,0.714515
3,2020,21,0 days 17:40:00,0.678519,0.714308


In [55]:
print("Total abnormal periods:", len(final_periods))
print("Total abnormal duration:", final_periods["duration"].sum())

Total abnormal periods: 252
Total abnormal duration: 8 days 03:55:00


In [56]:
model_data[["time", "anomaly_score", "anomaly_99"]].to_csv(
    "anomaly_scores.csv",
    index=False
)

print("Saved anomaly_scores.csv")
print("Rows:", len(model_data))

Saved anomaly_scores.csv
Rows: 374761
